In [ ]:
# -- Cell 1 -- rclone + Drive. Same pattern as the other notebooks.# Requires: Settings -> Internet ON, Accelerator GPU T4 x2, RCLONE_DRIVE_TOKEN.import os, subprocessr = subprocess.run("curl -s https://rclone.org/install.sh | sudo bash", shell=True)if r.returncode not in (0, 3):    raise RuntimeError("rclone install failed (exit %d)" % r.returncode)from kaggle_secrets import UserSecretsClienttoken = UserSecretsClient().get_secret("RCLONE_DRIVE_TOKEN")os.makedirs("/root/.config/rclone", exist_ok=True)with open("/root/.config/rclone/rclone.conf", "w") as f:    f.write("[drive]\ntype = drive\nscope = drive\ntoken = " + token + "\n")REMOTE = "drive:Distillation"out = subprocess.run("rclone lsf " + REMOTE, shell=True, capture_output=True, text=True)print(out.stdout or out.stderr)assert out.returncode == 0, "cannot see " + REMOTE

In [ ]:
# -- Cell 2 -- deps.## peft IS needed here, unlike the full-finetune PAMPA notebook: this run freezes# the backbone and trains LoRA adapters instead.subprocess.run('pip install -q -U "transformers>=5.0" "lightning>=2.4" "peft>=0.17" rdkit',               shell=True, check=True)# peft >=0.17 routes every LoRA layer through is_torchao_available(), which RAISES# rather than returning False when torchao is installed but older than 0.16.0 --# and Kaggle ships 0.10.0. Nothing here quantizes, so removing torchao is the# minimal fix; upgrading it would drag in a new torch build.subprocess.run("pip uninstall -y -q torchao", shell=True)from peft.import_utils import is_torchao_availableassert is_torchao_available() is False, "torchao still on the path -- LoRA will fail"import torch, numpy as np, pandas as pd, glob, json, time, importlibmod = importlib.import_module("transformers.tokenization_utils_tokenizers")assert hasattr(mod, "TokenizersBackend"), "transformers too old"print("torchao clear | GPUs", torch.cuda.device_count())for i in range(torch.cuda.device_count()):    p = torch.cuda.get_device_properties(i)    print("   cuda:%d %s %.0f GB" % (i, p.name, p.total_memory / 1e9))assert torch.cuda.device_count() >= 2, \    "this notebook pins one cluster per GPU -- set Accelerator to T4 x2"

In [ ]:
# -- Cell 3 -- pull code, their repo, and the student init.## NOTE: the distillation checkpoints are NOT needed. This run finetunes only the# warm-start model (their released peptideclm-2-mlm-small, untouched), so the# ~370 MB of latest.pt files can stay on Drive.WORK = "/kaggle/working"CODE, REPO, INIT = WORK + "/distill", WORK + "/their_repo", WORK + "/init"def pull(remote, local, extra=""):    os.makedirs(local, exist_ok=True)    subprocess.run("rclone copy %s/%s %s --transfers 16 %s -P" % (REMOTE, remote, local, extra),                   shell=True, check=True)if not os.path.exists(CODE + "/run_pampa.py"):    pull("distill", CODE)for sub in ("data", "training"):    if not os.path.isdir(REPO + "/" + sub):        pull("their_repo/" + sub, REPO + "/" + sub)if not os.path.exists(INIT + "/model.safetensors"):    pull("models/peptideclm-2-mlm-small", INIT)LORA_PY = CODE + "/pampa_lora.py"DATA = REPO + "/data/PAMPA_clusters.csv"for p in (CODE + "/run_pampa.py", LORA_PY, CODE + "/student.py", DATA,          INIT + "/model.safetensors"):    assert os.path.exists(p), "missing: " + p    print("ok", p.replace(WORK + "/", ""))df = pd.read_csv(DATA)print("\nPAMPA %d molecules | clusters %s"      % (len(df), df.cluster.value_counts().sort_index().to_dict()))

In [ ]:
# -- Cell 4 -- export the warm-start model as a plain HuggingFace directory.## export_student.py always writes the pristine init first, then looks for# treatment/control checkpoints and skips any it cannot find -- so pointing --runs# at a directory that does not exist is fine and yields warm-start only.## The folder name MUST contain "-small": their resolve_* helpers substring-match# the model name to choose batch size (16) and the full-finetune LR. pampa_lora.py# overrides the LR anyway, but batch size still comes from the name.EXPORT = WORK + "/exported"r = subprocess.run(["python", "export_student.py", "--runs", WORK + "/no_runs",                    "--init", INIT, "--out", EXPORT],                   cwd=CODE, capture_output=True, text=True)print(r.stdout[-2000:])if r.returncode != 0:    print(r.stderr[-2000:])assert r.returncode == 0, "export failed"WARM = EXPORT + "/peptideclm-2-mlm-small-warmstart"assert os.path.exists(WARM + "/model.safetensors")print("\nwarm-start ready:", WARM)

In [ ]:
# -- Cell 5 -- LoRA finetuning. One CLUSTER per GPU.##   GPU 0  cluster 1   (test fold 0, 1546 molecules held out)#   GPU 1  cluster 6   (test fold 5,  494 molecules held out)## 10 jobs total, 5 per card -- each cluster has 5 inner validation folds whose# predictions are ensembled, matching their nested-CV protocol.## --split-by cluster (not model) because there is only ONE model in this run, so# splitting by model would leave GPU 1 idle.## WHAT DIFFERS FROM THE FULL-FINETUNE RUN: only the trainable parameter set.# Backbone frozen, LoRA r=16 alpha=32 on qkv_proj (~2.2% of weights trainable),# lifted verbatim from their own classification config. Head, loss, optimizer,# schedule, batch size, early stopping and fold logic are all identical to their# regression script.## LR is 3e-4, not the full-finetune 1e-5: LoRA adapters initialise at zero and must# travel much further than a pretrained weight. 3e-4 is what their LoRA# classification script uses.## Budget: 100 epochs / 36000 steps. In pampa_lora.py the LR horizon is# min(epochs*steps_per_epoch, max_steps), so the two stay coupled. Keep max_steps# above one epoch (~292 steps) or training stops mid-epoch, val_rmse is never# logged, and EarlyStopping raises.OUT = WORK + "/pampa_lora"r = subprocess.run(["python", "-u", "run_pampa.py",                    "--repo-root", REPO, "--script", LORA_PY, "--data-csv", DATA,                    "--models", WARM,                    "--out", OUT, "--clusters", "1", "6", "--gpus", "0", "1",                    "--split-by", "cluster", "--seed", "101",                    "--max-epochs", "100", "--max-steps", "36000", "--patience", "20"],                   cwd=CODE)print("exit", r.returncode)

In [ ]:
# -- Cell 6 -- results, and the comparison that matters.res_path = OUT + "/pampa_metrics.csv"if os.path.exists(res_path):    res = pd.read_csv(res_path)    pd.set_option("display.width", 200)    print("LoRA finetuning, warm-start model:")    print(res.to_string(index=False))else:    print("no metrics written -- check", OUT + "/gpu*.log")print("""For reference, the SAME model under FULL finetuning (previous runs):    cluster 1    run1 R2 0.060    run2 R2 0.066    cluster 6    run1 R2 0.296    run2 R2 0.320Published reference (cluster-held-out R2): 32M MLM 0.13 | 32M MTR 0.38 | 337M 0.58WHY THIS RUN IS WORTH DOING: with the backbone frozen, downstream performance isgoverned by the quality of the FROZEN representation. Full finetuning can reshapea mediocre representation given enough capacity and steps, which masks differencesbetween backbones. LoRA cannot. This establishes the frozen-feature baseline forthe warm-start model; running the distilled model the same way is what would showwhether distillation improved the representation itself.""")subprocess.run("rclone copy %s %s/results/pampa_lora --drive-chunk-size 64M -P"               % (OUT, REMOTE), shell=True, check=True)print("uploaded to " + REMOTE + "/results/pampa_lora")